# SHAP-based Explainability Analysis for PatchXFormer

This notebook provides comprehensive SHAP analysis for understanding which weather parameters contribute most to solar power predictions.

Based on methodology from: "Solar energy prediction through machine learning models" (PLOS ONE, 2025)

## 1. Setup and Imports

In [ ]:
# Install required packages if not available
!pip install shap -q
!pip install lime -q

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"PyTorch version: {torch.__version__}")
print(f"SHAP version: {shap.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Configuration

In [ ]:
# Configuration - Modify these settings as needed
class Config:
    # Dataset settings
    dataset = 'sl_piliyandala'
    root_path = f'./dataset/{dataset}'
    
    # Model settings
    model_name = 'PatchXFormer'
    task_name = 'long_term_forecast'
    seq_len = 96
    label_len = 48
    pred_len = 96
    
    # Model architecture
    enc_in = 10
    dec_in = 10
    c_out = 10
    d_model = 256
    n_heads = 8
    e_layers = 2
    d_layers = 1
    d_ff = 512
    dropout = 0.1
    embed = 'timeF'
    activation = 'gelu'
    factor = 3
    
    # Additional params
    features = 'M'
    target = 'Solar Power Output'
    freq = 'h'
    expand = 2
    d_conv = 4
    distil = True
    des = 'Exp'
    
    # Hardware
    use_gpu = True
    gpu = 0
    num_workers = 0
    batch_size = 32
    
    # Checkpoints
    checkpoints = './checkpoints/'
    
    # SHAP settings
    num_samples = 100
    background_samples = 50
    output_dir = f'./shap_results/{dataset}_pred{pred_len}'

# Feature names for solar dataset
FEATURE_NAMES = [
    'dayofyear', 'timeofday', 'temp', 'dew', 
    'humidity', 'winddir', 'windspeed', 'pressure', 
    'cloudcover', 'Solar Power Output'
]

WEATHER_FEATURES = ['temp', 'dew', 'humidity', 'winddir', 'windspeed', 'pressure', 'cloudcover']

# Create output directory
os.makedirs(Config.output_dir, exist_ok=True)

print("Configuration loaded!")
print(f"Dataset: {Config.dataset}")
print(f"Prediction length: {Config.pred_len}")
print(f"Output directory: {Config.output_dir}")

## 3. Load Model and Data

In [ ]:
# Import project modules
from data_provider.data_factory import data_provider
from exp.exp_basic import Exp_Basic
from models import PatchXFormer

# Create args-like object
class Args:
    pass

args = Args()
for key, value in vars(Config).items():
    if not key.startswith('_'):
        setattr(args, key, value)

# Additional required args
args.data = 'custom'
args.data_path = 'solar.csv'
args.model_id = f'solar_{Config.dataset}{Config.seq_len}_{Config.pred_len}'
args.model = Config.model_name
args.inverse = False
args.use_multi_gpu = False
args.device_ids = [0]

# Set device
if torch.cuda.is_available() and args.use_gpu:
    device = torch.device(f'cuda:{args.gpu}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

args.device = device
print(f"Using device: {device}")

In [ ]:
# Load model
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

exp = Exp_Long_Term_Forecast(args)
model = exp._build_model().to(device)

# Generate setting string
setting = '{}_{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_expand{}_dc{}_fc{}_eb{}_dt{}_{}_{}'.format(
    args.task_name, args.model_id, args.model, args.data,
    args.features, args.seq_len, args.label_len, args.pred_len,
    args.d_model, args.n_heads, args.e_layers, args.d_layers,
    args.d_ff, args.expand, args.d_conv, args.factor,
    args.embed, args.distil, args.des, 0
)

# Load checkpoint if exists
checkpoint_path = os.path.join(args.checkpoints, setting, 'checkpoint.pth')
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from: {checkpoint_path}")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("Checkpoint loaded successfully!")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")
    print("Using randomly initialized model for demonstration")

model.eval()
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Load test data
_, test_loader = exp._get_data(flag='test')
print(f"Test batches: {len(test_loader)}")

# Get sample batch for inspection
for batch_x, batch_y, batch_x_mark, batch_y_mark in test_loader:
    print(f"Input shape (x_enc): {batch_x.shape}")
    print(f"Target shape (y): {batch_y.shape}")
    print(f"Time marks shape: {batch_x_mark.shape}")
    break

## 4. Collect Data for SHAP Analysis

In [ ]:
# Collect samples from test loader
num_samples = Config.num_samples + Config.background_samples

all_inputs = []
all_marks = []
all_targets = []

for batch_x, batch_y, batch_x_mark, batch_y_mark in test_loader:
    all_inputs.append(batch_x.numpy())
    all_marks.append(batch_x_mark.numpy())
    all_targets.append(batch_y.numpy())
    if len(all_inputs) * batch_x.shape[0] >= num_samples:
        break

all_inputs = np.concatenate(all_inputs, axis=0)[:num_samples]
all_marks = np.concatenate(all_marks, axis=0)[:num_samples]
all_targets = np.concatenate(all_targets, axis=0)[:num_samples]

print(f"Collected {len(all_inputs)} samples for analysis")
print(f"Input shape: {all_inputs.shape}")

In [ ]:
# Aggregate features across time steps (mean pooling)
aggregated_inputs = all_inputs.mean(axis=1)
print(f"Aggregated input shape: {aggregated_inputs.shape}")

# Split into background and explain data
background_data = aggregated_inputs[:Config.background_samples]
explain_data = aggregated_inputs[Config.background_samples:Config.background_samples + Config.num_samples]

print(f"Background samples: {len(background_data)}")
print(f"Samples to explain: {len(explain_data)}")

## 5. Create Model Wrapper for SHAP

In [ ]:
def model_predict_aggregated(x_aggregated):
    """
    Wrapper for model prediction using aggregated features.
    Replicates aggregated features across time steps for model input.
    """
    model.eval()
    with torch.no_grad():
        batch_size = x_aggregated.shape[0]
        
        # Replicate aggregated features across time steps
        x_enc = np.tile(x_aggregated[:, np.newaxis, :], (1, args.seq_len, 1))
        x_enc = torch.FloatTensor(x_enc).to(device)
        
        # Create dummy time marks and decoder inputs
        x_mark_enc = torch.zeros(batch_size, args.seq_len, 4).to(device)
        dec_inp = torch.zeros(batch_size, args.label_len + args.pred_len,
                             x_enc.shape[-1]).to(device)
        x_mark_dec = torch.zeros(batch_size, args.label_len + args.pred_len, 4).to(device)
        
        outputs = model(x_enc, x_mark_enc, dec_inp, x_mark_dec)
        
        # Return mean prediction of target variable (last column)
        return outputs[:, :, -1].mean(dim=1).cpu().numpy()

# Test the wrapper
test_pred = model_predict_aggregated(background_data[:5])
print(f"Test prediction shape: {test_pred.shape}")
print(f"Test predictions: {test_pred}")

## 6. Compute SHAP Values

In [ ]:
%%time
# Initialize SHAP KernelExplainer
print("Initializing SHAP KernelExplainer...")
explainer = shap.KernelExplainer(model_predict_aggregated, background_data)

print(f"Computing SHAP values for {len(explain_data)} samples...")
print("This may take several minutes...")

shap_values = explainer.shap_values(explain_data, nsamples=100)
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# Save SHAP values
np.save(os.path.join(Config.output_dir, 'shap_values.npy'), shap_values)
np.save(os.path.join(Config.output_dir, 'explain_data.npy'), explain_data)
print(f"SHAP values saved to {Config.output_dir}")

## 7. Feature Importance Analysis

In [ ]:
# Compute feature importance (mean |SHAP|)
feature_importance = np.abs(shap_values).mean(axis=0)

# Create importance DataFrame
importance_df = pd.DataFrame({
    'Feature': FEATURE_NAMES[:len(feature_importance)],
    'Mean_Abs_SHAP': feature_importance
}).sort_values('Mean_Abs_SHAP', ascending=False)

print("\n" + "="*50)
print("FEATURE IMPORTANCE RANKING")
print("="*50)
print(importance_df.to_string(index=False))
print("="*50)

# Save to CSV
importance_df.to_csv(os.path.join(Config.output_dir, 'feature_importance.csv'), index=False)

In [ ]:
# Plot Feature Importance Bar Chart (Similar to Figure 7a in PLOS ONE paper)
fig, ax = plt.subplots(figsize=(10, 6))

sorted_idx = np.argsort(feature_importance)[::-1]
sorted_features = [FEATURE_NAMES[i] for i in sorted_idx]
sorted_importance = feature_importance[sorted_idx]

colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(sorted_features)))

bars = ax.barh(range(len(sorted_features)), sorted_importance, color=colors)
ax.set_yticks(range(len(sorted_features)))
ax.set_yticklabels(sorted_features)
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.set_title('Feature Importance - Mean Absolute SHAP Values\nPatchXFormer Solar Power Forecasting', fontsize=14)

# Add value labels
for bar, val in zip(bars, sorted_importance):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, 
            f'{val:.4f}', va='center', fontsize=10)

ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(Config.output_dir, 'feature_importance_bar.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(Config.output_dir, 'feature_importance_bar.pdf'), format='pdf', bbox_inches='tight')
plt.show()

## 8. SHAP Summary Plot (Global Interpretation)

In [ ]:
# SHAP Summary Plot (Similar to Figure 7b in PLOS ONE paper)
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values, 
    explain_data,
    feature_names=FEATURE_NAMES[:shap_values.shape[1]],
    show=False,
    plot_size=(12, 8)
)
plt.title('SHAP Summary Plot - PatchXFormer Solar Forecasting', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(Config.output_dir, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(Config.output_dir, 'shap_summary_plot.pdf'), format='pdf', bbox_inches='tight')
plt.show()

## 9. Partial Dependence Plots

In [ ]:
# Create partial dependence plots for all features (Similar to Figure 8 in PLOS ONE paper)
n_features = min(shap_values.shape[1], len(FEATURE_NAMES))

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx in range(n_features):
    feature_name = FEATURE_NAMES[idx]
    ax = axes[idx]
    
    feature_values = explain_data[:, idx]
    shap_feature = shap_values[:, idx]
    
    # Scatter plot with trend line
    ax.scatter(feature_values, shap_feature, alpha=0.5, s=20, c='steelblue')
    
    # Add smoothed trend line
    try:
        from scipy.ndimage import gaussian_filter1d
        sorted_idx = np.argsort(feature_values)
        x_sorted = feature_values[sorted_idx]
        y_sorted = shap_feature[sorted_idx]
        y_smooth = gaussian_filter1d(y_sorted, sigma=5)
        ax.plot(x_sorted, y_smooth, 'r-', linewidth=2, label='Trend')
    except:
        pass
    
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel(feature_name, fontsize=10)
    ax.set_ylabel('SHAP Value', fontsize=10)
    ax.set_title(f'{feature_name}', fontsize=12)

plt.suptitle('Partial SHAP Dependence Plots - All Features\nPatchXFormer Solar Forecasting', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(Config.output_dir, 'partial_dependence_all.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(Config.output_dir, 'partial_dependence_all.pdf'), format='pdf', bbox_inches='tight')
plt.show()

## 10. Weather Parameter Contribution Analysis

In [ ]:
# Focus on weather parameters only
weather_indices = [i for i, name in enumerate(FEATURE_NAMES[:shap_values.shape[1]]) 
                   if name in WEATHER_FEATURES]
weather_names = [FEATURE_NAMES[i] for i in weather_indices]

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.flatten()

analysis_results = []

for i, (idx, name) in enumerate(zip(weather_indices, weather_names)):
    if i >= len(axes):
        break
        
    ax = axes[i]
    
    feature_vals = explain_data[:, idx]
    shap_vals = shap_values[:, idx]
    
    # Create hexbin plot
    h = ax.hexbin(feature_vals, shap_vals, gridsize=20, cmap='RdYlBu_r', mincnt=1)
    ax.axhline(y=0, color='white', linestyle='--', alpha=0.7)
    
    ax.set_xlabel(f'{name} Value', fontsize=10)
    ax.set_ylabel('SHAP Value', fontsize=10)
    ax.set_title(f'{name}', fontsize=12)
    
    plt.colorbar(h, ax=ax, label='Count')
    
    # Compute statistics
    mean_shap = np.abs(shap_vals).mean()
    std_shap = shap_vals.std()
    correlation = np.corrcoef(feature_vals, shap_vals)[0, 1]
    
    analysis_results.append({
        'Feature': name,
        'Mean |SHAP|': mean_shap,
        'Std SHAP': std_shap,
        'Correlation': correlation
    })

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Weather Parameter Contribution Analysis\nPatchXFormer Solar Power Forecasting', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(Config.output_dir, 'weather_contribution_analysis.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(Config.output_dir, 'weather_contribution_analysis.pdf'), format='pdf', bbox_inches='tight')
plt.show()

# Display weather statistics
weather_df = pd.DataFrame(analysis_results).sort_values('Mean |SHAP|', ascending=False)
weather_df.to_csv(os.path.join(Config.output_dir, 'weather_contribution_stats.csv'), index=False)

print("\n" + "="*60)
print("WEATHER PARAMETER CONTRIBUTION STATISTICS")
print("="*60)
print(weather_df.to_string(index=False))
print("="*60)

## 11. Local SHAP Explanations (Individual Predictions)

In [ ]:
# Local explanation for specific samples (Similar to Figure 9 in PLOS ONE paper)
sample_indices = [0, 1, 2]

for sample_idx in sample_indices:
    sample_shap = shap_values[sample_idx]
    sample_data = explain_data[sample_idx]
    
    plt.figure(figsize=(12, 8))
    
    # Sort by absolute SHAP value
    sorted_idx = np.argsort(np.abs(sample_shap))[::-1]
    
    colors = ['red' if v < 0 else 'blue' for v in sample_shap[sorted_idx]]
    
    y_pos = np.arange(len(sorted_idx))
    plt.barh(y_pos, sample_shap[sorted_idx], color=colors, alpha=0.7)
    plt.yticks(y_pos, [FEATURE_NAMES[i] for i in sorted_idx])
    
    plt.xlabel('SHAP Value (Impact on Prediction)', fontsize=12)
    plt.ylabel('Feature', fontsize=12)
    plt.title(f'Local SHAP Explanation - Sample {sample_idx}', fontsize=14)
    
    # Add feature values as annotations
    for i, idx in enumerate(sorted_idx):
        val = sample_data[idx]
        shap_val = sample_shap[idx]
        plt.annotate(f'{val:.2f}', 
                    xy=(shap_val, i), 
                    xytext=(5 if shap_val >= 0 else -5, 0),
                    textcoords='offset points',
                    va='center', ha='left' if shap_val >= 0 else 'right',
                    fontsize=9)
    
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    
    plt.savefig(os.path.join(Config.output_dir, f'local_explanation_sample_{sample_idx}.png'), dpi=300, bbox_inches='tight')
    plt.show()

## 12. Summary Report

In [ ]:
print("\n" + "="*70)
print("SHAP EXPLAINABILITY ANALYSIS SUMMARY")
print("="*70)
print(f"\nDataset: {Config.dataset}")
print(f"Model: PatchXFormer")
print(f"Samples analyzed: {len(explain_data)}")
print(f"\nOutput saved to: {Config.output_dir}")

print("\n" + "-"*70)
print("TOP 5 MOST IMPORTANT FEATURES:")
print("-"*70)
for i, row in importance_df.head(5).iterrows():
    print(f"  {importance_df.index.get_loc(i)+1}. {row['Feature']:20s} Mean|SHAP| = {row['Mean_Abs_SHAP']:.4f}")

print("\n" + "-"*70)
print("TOP WEATHER PARAMETERS:")
print("-"*70)
for i, row in weather_df.head(5).iterrows():
    print(f"  {i+1}. {row['Feature']:15s} Mean|SHAP| = {row['Mean |SHAP|']:.4f}, Correlation = {row['Correlation']:.3f}")

print("\n" + "-"*70)
print("FILES GENERATED:")
print("-"*70)
for f in os.listdir(Config.output_dir):
    print(f"  - {f}")
print("="*70)

## 13. Conclusion

This SHAP analysis reveals which weather parameters contribute most to the PatchXFormer model's solar power predictions. 

Key findings based on the analysis:
- Features ranked by their mean absolute SHAP values indicate their overall importance
- Partial dependence plots show the non-linear relationship between features and predictions
- Local explanations provide insight into individual prediction decisions

These results can be used for:
1. Understanding model behavior and building trust
2. Feature selection and engineering
3. Thesis/paper figures (use PDF exports for publication quality)